In [1]:
import pennylane as qp
import numpy as np
import matplotlib.pyplot as plt

In [2]:
num_wires = 3
dev = qp.device("default.qubit", wires=num_wires)

In [3]:
@qp.qnode(dev)
def make_basis_state(basis_id):
    """Produce the 3-qubit basis state corresponding to |basis_id>.

    Note that the system starts in |000>.

    Args:
        basis_id (int): An integer value identifying the basis state to construct.

    Returns:
        np.array[complex]: The computational basis state |basis_id>.
    """

    # CREATE THE BASIS STATE
    rep = np.binary_repr(basis_id, num_wires)
    for i, s in enumerate(rep):
        if s == "1": qp.PauliX(i)

    return qp.state()

In [4]:
basis_id = 3
print(f"Output state = {make_basis_state(basis_id)}")

Output state = [0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]


In [5]:
# Creates a device with *two* qubits
dev = qp.device("default.qubit", wires=2)

In [6]:
@qp.qnode(dev)
def two_qubit_circuit():
    # PREPARE |+>|1>
    qp.Hadamard(0)
    qp.PauliX(1)

    # RETURN TWO EXPECTATION VALUES, Y ON FIRST QUBIT, Z ON SECOND QUBIT
    return (qp.expval(qp.Y(0)), qp.expval(qp.Z(1)))

In [7]:
@qp.qnode(dev)
def create_one_minus():
    # PREPARE |1>|->
    qp.PauliX(0)
    qp.PauliX(1)
    qp.Hadamard(1)

    # RETURN A SINGLE EXPECTATION VALUE Z \otimes X
    return qp.expval(qp.Z(0) @ qp.X(1))

In [8]:
print(create_one_minus())

0.9999999999999996


In [9]:
@qp.qnode(dev)
def circuit_1(theta):
    """Implement the circuit and measure Z I and I Z.

    Args:
        theta (float): a rotation angle.

    Returns:
        float, float: The expectation values of the observables Z I, and I Z
    """
    qp.RX(theta, 0)
    qp.RY(theta * 2, 1)

    return (qp.expval(qp.Z(0)), qp.expval(qp.Z(1)))

In [10]:
@qp.qnode(dev)
def circuit_2(theta):
    """Implement the circuit and measure Z Z.

    Args:
        theta (float): a rotation angle.

    Returns:
        float: The expectation value of the observable Z Z
    """

    qp.RX(theta, 0)
    qp.RY(theta * 2, 1)

    return (qp.expval(qp.Z(0) @ qp.Z(1)))

In [11]:
def zi_iz_combination(ZI_results, IZ_results):
    """Implement a function that acts on the ZI and IZ results to
    produce the ZZ results. How do you think they should combine?

    Args:
        ZI_results (np.array[float]): Results from the expectation value of
            ZI in circuit_1.
        IZ_results (np.array[float]): Results from the expectation value of
            IZ in circuit_2.

    Returns:
        np.array[float]: A combination of ZI_results and IZ_results that
        produces results equivalent to measuring ZZ.
    """

    combined_results = ZI_results * IZ_results

    return combined_results

In [12]:
theta = np.linspace(0, 2 * np.pi, 100)

# Run circuit 1, and process the results
circuit_1_results = np.array([circuit_1(t) for t in theta])

ZI_results = circuit_1_results[:, 0]
IZ_results = circuit_1_results[:, 1]
combined_results = zi_iz_combination(ZI_results, IZ_results)

# Run circuit 2
ZZ_results = np.array([circuit_2(t) for t in theta])

# Plot your results
# plot = plotter(theta, ZI_results, IZ_results, ZZ_results, combined_results)